In [ ]:
import torch
import tiktoken
from configs.model import ModelConfig
from configs.training import TrainingConfig
from configs.scheduler import SchedulerConfig
from configs.optimizer import OptimizerConfig
from configs.checkpoint import CheckpointConfig
from models.qwen import Qwen
from generation.sample_text import generate, generate_sample_text, text_to_token_ids,token_ids_to_text
from datasets.download_data.the_verdict.verdict import download_the_verdict
from datasets.preprocess import prepare_dataset
from datasets.dataloader import createDataLoader
from evaluation.losses import cross_entropy_loss,token_accuracy
from trainer.trainer import Trainer

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

In [10]:
Qwen_SMALL = ModelConfig(
    emb_dim=96,
    n_layers=2,
    n_heads=12,
    kv_heads=6,
    activation="gelu",
    context_length=24
)


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Qwen(Qwen_SMALL).to(device)

### Text Generation

In [8]:
text = "Every effort moves you "
tokenized_text = text_to_token_ids(text, tokenizer)
ids = generate(model, tokenized_text, max_new_tokens=20, context_size=Qwen_SMALL.context_length)
response = token_ids_to_text(ids, tokenizer)
print(response)

Every effort moves you  Machine


In [9]:
res_text = generate_sample_text(model,device='cpu',tokenizer=tokenizer,text=text,max_new_tokens=20)
print(res_text)

Every effort moves you  Machine parliamentary graduationerm ceasesStudies Sutton Missonian exerc revolutionsiding Spoonuebl screamedotine 4000 conclusionSleep ori


### Training

In [14]:
TRAIN_CONFIG = TrainingConfig(
    epoch=1,
    batch_size=2,
    stride=24,
    context_length=24
)

OPTIMIZER_CONFIG = OptimizerConfig()
SCHEDULER_CONFIG = SchedulerConfig()
CHECKPOINT_CONFIG = CheckpointConfig()

In [ ]:
raw_text,verdict_dir = download_the_verdict()
train_ids, val_ids = prepare_dataset(raw_text, tokenizer,TRAIN_CONFIG,verdict_dir)
train_dataloader = createDataLoader(train_ids, TRAIN_CONFIG)
val_dataloader = createDataLoader(val_ids, TRAIN_CONFIG)

TrainingConfig(epoch=1, batch_size=2, stride=24, context_length=24, learning_rate=0.0003, weight_decay=0.1, grad_clip=1.0, mixed_precision=False, shuffle=False, num_workers=0, drop_last=True, gradient_accumulation_steps=1, train_data_ratio=0.9)
TrainingConfig(epoch=1, batch_size=2, stride=24, context_length=24, learning_rate=0.0003, weight_decay=0.1, grad_clip=1.0, mixed_precision=False, shuffle=False, num_workers=0, drop_last=True, gradient_accumulation_steps=1, train_data_ratio=0.9)


In [16]:
trainer=Trainer(model,tokenizer,train_dataloader,val_dataloader,device,cross_entropy_loss,token_accuracy,CHECKPOINT_CONFIG,TRAIN_CONFIG,OPTIMIZER_CONFIG,SCHEDULER_CONFIG)

In [17]:
trainer.fit()           ### use arg resume_latest=True or resume_best=True to resume training

100%|██████████| 96/96 [00:23<00:00,  4.13it/s]


Checkpoint saved -> checkpoints\checkpoint_96.pt
Best checkpoint saved -> checkpoints\best_checkpoint.pt
Output text:
 Every effort moves you Spring
after 1 epoch global step 96 the train loss 10.152530312538147 val loss 8.48401927947998 and train acc| 0.04600694484543055 val acc| 0.0416666716337204 


In [21]:
trainer.metrics.metrics

defaultdict(list,
            {'train_loss': [10.152530312538147],
             'train_acc': [0.04600694484543055],
             'train_ppl': [25655.937685888923],
             'val_loss': [8.48401927947998],
             'val_acc': [0.0416666716337204],
             'val_ppl': [4836.851538578296]})